# OpenSearch: how the Knowledge Base searches

Local cluster from `docker-compose.yml` (security plugin disabled, no auth, published on port 9201). The app searches the `items` index.

In [12]:
from opensearchpy import OpenSearch

client = OpenSearch("http://localhost:9201")
print(client.info()["version"]["number"])

3.9.0


## 1 · Keyword search (BM25)

The app's query matches the words in title (weighted ×2), summary, key points, a Thought's text and the Owner's note.

In [19]:
# QUERY = "dron"
QUERY = "formula"

keyword_query = {
    "multi_match": {"query": QUERY, "fields": ["title^2", "summary", "key_points", "text", "annotations.notes"]}
}
keyword_resp = client.search(
    index="items",
    # body={"query": keyword_query, "_source": ["title", "normalised_url", "tags"], "size": 20},  # skip the big embedding vector
    body={"query": keyword_query, "_source": ["title", "normalised_url", "tags", "embedding"], "size": 20}, # show also embedding 
    # body={"query": keyword_query, "_source": ["title", "normalised_url", "tags", "summary", "key_points", "text"], "size": 20} # show text
)


def show(resp):
    for hit in resp["hits"]["hits"][:5]:
        print(round(hit["_score"], 3), hit["_source"]["title"])


show(keyword_resp)

2.478 Mercedes' Kimi Antonelli reveals what race number he will use to defend Formula 1 title in 2027


In [20]:
keyword_resp["hits"]["hits"][0]

{'_index': 'items',
 '_id': '98e0b57d8ca2e63183f7328ecce302955ec7301f259803005753ed591377e0ed',
 '_score': 2.4778748,
 '_source': {'normalised_url': 'https://www.formula1.com/en/latest/article/antonelli-reveals-what-race-number-he-will-use-to-defend-formula-1-title-in-2027.3GBWpLc1FUNDolu43D8Llu',
  'embedding': [0.02015686,
   -0.03161621,
   0.045959473,
   -0.006095886,
   -0.018966675,
   -0.017913818,
   0.0008635521,
   0.045776367,
   0.01235199,
   -0.039276123,
   -0.037353516,
   0.026123047,
   -0.056884766,
   -0.0107803345,
   0.053131104,
   0.010124207,
   -0.0015573502,
   -0.007865906,
   -0.00623703,
   -0.026062012,
   -0.009384155,
   0.025131226,
   0.008865356,
   0.025054932,
   0.052246094,
   -0.001876831,
   -0.04949951,
   -0.023986816,
   0.014663696,
   0.05218506,
   0.013420105,
   -0.009811401,
   -0.008705139,
   -0.024902344,
   0.028274536,
   -0.020370483,
   -0.00440979,
   -0.045654297,
   -0.0140686035,
   -0.010612488,
   -0.011154175,
   -0.0001

## 2 · Vector search (k-NN)

The query is embedded by the same model as the Items (1536 numbers), and OpenSearch returns the nearest Items by cosine similarity.

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
vector = OpenAI().embeddings.create(model="text-embedding-3-small", input=QUERY).data[0].embedding  # paid call
print(len(vector), vector[:3])

knn_query = {"knn": {"embedding": {"vector": vector, "k": 50}}}
vector_resp = client.search(index="items", body={"query": knn_query, "_source": ["title"], "size": 20})
show(vector_resp)

1536 [-0.03826904296875, 0.0173797607421875, -0.030975341796875]
0.62 test poznamky
0.606 personal note 29.10.
0.596 SINGAPORE LOWDOWN: All the key moments as Max Verstappen dominates and Ferrari face strategy turmoil
0.591 Cerna AI call 16.10
0.583 Další komplikace pro Russella. V Singapuru ho čeká těžká šichta


In [5]:
# vector

## 3 · Hybrid search

Both queries run at once. The `kb-hybrid` search pipeline scales each result list to 0–1 (min-max) and averages them 50/50.

In [6]:
client.transport.perform_request("GET", "/_search/pipeline/kb-hybrid") # mame nadefinovane pri zakladani indexu, funguje na urovni db.

{'kb-hybrid': {'description': 'kb hybrid search: min-max normalised BM25 + k-NN, weighted arithmetic mean',
  'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'},
     'combination': {'technique': 'arithmetic_mean',
      'parameters': {'weights': [0.5, 0.5]}}}}]}}

In [7]:
hybrid_resp = client.search(
    index="items",
    body={"query": {"hybrid": {"queries": [keyword_query, knn_query]}}, "_source": ["title"], "size": 5},
    params={"search_pipeline": "kb-hybrid"},
)


def rank(resp, item_id):
    ids = [hit["_id"] for hit in resp["hits"]["hits"]]
    return ids.index(item_id) + 1 if item_id in ids else "–"


print("hybrid  keyword  vector  title")
for hit in hybrid_resp["hits"]["hits"]:
    print(f"{hit['_score']:.3f}  {rank(keyword_resp, hit['_id'])!s:>7}  {rank(vector_resp, hit['_id'])!s:>6}  {hit['_source']['title']}")

hybrid  keyword  vector  title
0.500        1      11  Mercedes' Kimi Antonelli reveals what race number he will use to defend Formula 1 title in 2027
0.500        –       1  test poznamky
0.433        –       2  personal note 29.10.
0.385        –       3  SINGAPORE LOWDOWN: All the key moments as Max Verstappen dominates and Ferrari face strategy turmoil
0.360        –       4  Cerna AI call 16.10


## 4 · How the data are stored

`items` is the current view of each Item (what is searched); `versions` keeps every Capture, never updated.

In [8]:
# mapping = client.indices.get_mapping(index="items")["items"]["mappings"]["properties"]
# {field: spec.get("type", "object") for field, spec in mapping.items()}, mapping["embedding"]

In [9]:
item_id = keyword_resp["hits"]["hits"][0]["_id"]  # sha256 of the normalised URL
item = client.get(index="items", id=item_id)["_source"]
item["embedding"] = f"{len(item['embedding'])} floats {item['embedding'][:3]} …"
item

{'id': '98e0b57d8ca2e63183f7328ecce302955ec7301f259803005753ed591377e0ed',
 'normalised_url': 'https://www.formula1.com/en/latest/article/antonelli-reveals-what-race-number-he-will-use-to-defend-formula-1-title-in-2027.3GBWpLc1FUNDolu43D8Llu',
 'original_url': 'https://www.formula1.com/en/latest/article/antonelli-reveals-what-race-number-he-will-use-to-defend-formula-1-title-in-2027.3GBWpLc1FUNDolu43D8Llu',
 'media_type': 'article',
 'first_captured_at': '2026-11-03T09:37:16.756000Z',
 'captured_at': '2026-11-03T09:38:07.005000Z',
 'current_version_id': '98e0b57d8ca2e63183f7328ecce302955ec7301f259803005753ed591377e0ed:2026-11-03T09:38:07.005Z',
 'title': "Mercedes' Kimi Antonelli reveals what race number he will use to defend Formula 1 title in 2027",
 'summary': 'Kimi Antonelli, the newly crowned Formula 1 World Champion, has announced that he will continue to use race number 12 to defend his title in 2027. Antonelli, who became the youngest F1 champion at age 20 after finishing third

In [10]:
client.search(
    index="versions",
    body={"query": {"term": {"item_id": item_id}}, "_source": {"excludes": ["enrichment.embedding"]}},
)["hits"]["hits"]

[{'_index': 'versions',
  '_id': '98e0b57d8ca2e63183f7328ecce302955ec7301f259803005753ed591377e0ed:2026-11-03T09:37:16.756Z',
  '_score': 1.0,
  '_source': {'enrichment': {'summary': 'Kimi Antonelli, the newly crowned Formula 1 World Champion, has announced that he will continue to use race number 12 to defend his title in 2027. Antonelli, who became the youngest F1 champion at age 20 after finishing third in the Mexico City Grand Prix, achieved nine wins during the season and only missed the podium in three races. He expressed that the number 12 holds special significance for him, as it was also the number used by his idol Ayrton Senna. Antonelli mentioned his superstition regarding the number 1, stating that it has brought him bad luck in the past, which influenced his decision to stick with number 12.',
    'tag_model': 'gpt-4o-mini',
    'tag_prompt_version': '1',
    'embedding_model': 'text-embedding-3-small',
    'taxonomy_version': 13,
    'language': 'en',
    'model': 'gpt-4o